# Prophet mit Wetter + Co-Schadstoffen als **Lag-Regressoren**

Eigenständiges Notebook (kein anderes muss vorher laufen). **Kernidee:** Zur Vorhersagezeit werden **keine Testwerte** der Regressoren benutzt — weder Wetter noch Schadstoffe fließen mit ihren Zukunfts-/Testwerten ein. Stattdessen wird der **letzte bekannte Wert eingefroren** („gelaggt"):

- **12.4 Einzelanalyse:** eingefroren auf den letzten **Trainingswert** → ausschließlich Trainingsdaten.
- **12.5 rollierend (12 Stationen):** eingefroren am jeweiligen **Startpunkt** → nie das Vorhersagefenster.

Verglichen wird je Horizont (8/24/48/72 h): **Seasonal Naive**, **Prophet + Wetter (Lag)** und **Prophet + Wetter+Co (Lag)** — so sieht man den Zusatznutzen der Schadstoffe.

> ⚙️ Kernel `ts-tutorial` (dort ist Prophet installiert). Voraussetzung: `../data/prepared/behandelt/` und `../data/PRSA_Data_20130301-20170228/` vorhanden.

## 1. Setup: Imports, Konfiguration, Datenlader

In [1]:
import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
from prophet import Prophet

# --- Konfiguration (identisch zum Hauptnotebook) ---
PREP     = Path('../data/prepared')
RAW_DIR  = Path('../data/PRSA_Data_20130301-20170228')
STATION  = 'Aotizhongxin'
REGRESSOREN = ['TEMP', 'DEWP', 'PRES', 'WSPM', 'RAIN', 'wd_sin', 'wd_cos']   # Wetter
CO_POLL     = ['PM10', 'SO2', 'NO2', 'CO', 'O3']                             # Co-Schadstoffe
REGS_WX  = REGRESSOREN
REGS_ALL = REGRESSOREN + CO_POLL
HORIZONTE = [8, 24, 48, 72]
PROPHET_KWARGS = dict(yearly_seasonality=True, weekly_seasonality=True,
                      daily_seasonality=True, changepoint_prior_scale=0.05)

# alle Stationsnamen automatisch
stationen = sorted(p.name.replace('prophet_train_', '').replace('.csv', '')
                   for p in (PREP / 'basis').glob('prophet_train_*.csv'))
print(f'{len(stationen)} Stationen:', stationen)
print('Prophet', __import__('prophet').__version__)

12 Stationen: ['Aotizhongxin', 'Changping', 'Dingling', 'Dongsi', 'Guanyuan', 'Gucheng', 'Huairou', 'Nongzhanguan', 'Shunyi', 'Tiantan', 'Wanliu', 'Wanshouxigong']
Prophet 1.3.0


In [2]:
# --- Datenlader & Hilfsfunktionen ---
def lade(variante, station=STATION):
    tr = pd.read_csv(PREP / variante / f'prophet_train_{station}.csv', parse_dates=['ds'])
    te = pd.read_csv(PREP / variante / f'prophet_test_{station}.csv',  parse_dates=['ds'])
    return tr, te

def lade_coschadstoffe(station):
    d = pd.read_csv(RAW_DIR / f'PRSA_Data_{station}_20130301-20170228.csv')
    d['ds'] = pd.to_datetime(d[['year', 'month', 'day', 'hour']])
    s = d[['ds'] + CO_POLL].sort_values('ds').set_index('ds')
    s = s.reindex(pd.date_range(s.index.min(), s.index.max(), freq='h'))
    s[CO_POLL] = s[CO_POLL].interpolate(method='time', limit_direction='both')
    return s.reset_index().rename(columns={'index': 'ds'})

def mit_coschadstoffe(df, station):
    out = df.merge(lade_coschadstoffe(station), on='ds', how='left')
    out[CO_POLL] = out[CO_POLL].interpolate().bfill().ffill()
    return out

def mae(y, yh):  return float(np.mean(np.abs(np.asarray(y, float) - np.asarray(yh, float))))
def rmse(y, yh): return float(np.sqrt(np.mean((np.asarray(y, float) - np.asarray(yh, float)) ** 2)))
def mape(y, yh):
    y = np.asarray(y, float); yh = np.asarray(yh, float); m = y > 0
    return float(np.mean(np.abs((y[m] - yh[m]) / y[m])) * 100) if m.any() else np.nan
def mase_skala(train_y, mp=24):
    a = np.asarray(train_y, float); return float(np.nanmean(np.abs(a[mp:] - a[:-mp])))

def fit_prophet(train, regressoren):
    m = Prophet(**PROPHET_KWARGS); m.add_country_holidays(country_name='CN')
    for r in regressoren: m.add_regressor(r)
    m.fit(train[['ds', 'y'] + regressoren]); return m

def predict_frozen(m, ds_index, werte_row, regressoren):
    """Alle Regressoren auf werte_row (ein bekannter Zeitpunkt) eingefroren -> kein Testwert."""
    fut = pd.DataFrame({'ds': ds_index})
    for r in regressoren: fut[r] = werte_row[r]
    return m.predict(fut)['yhat'].clip(lower=0).to_numpy()

def seasonal_naive(train, test, mp=24):
    prof = train.sort_values('ds').tail(mp).assign(h=lambda d: d['ds'].dt.hour).set_index('h')['y']
    return prof.reindex(test['ds'].dt.hour).to_numpy()

## 12.4 Einzelanalyse (Detailstation) — Regressoren **nur aus dem Training**

Modell wird mit den echten Trainings-Regressoren gefittet; für die Vorhersage werden Wetter **und** Schadstoffe auf den **letzten Trainingswert** eingefroren. Es fließt kein einziger Testwert der Regressoren ein.

> Hinweis: Der **Jahres**-Wert ist der Extremfall (Regressoren ein Jahr lang eingefroren) — aussagekräftig sind die **kurzen** Horizonte.

In [3]:
tr, te = lade('behandelt', STATION)
tr = mit_coschadstoffe(tr, STATION); te = mit_coschadstoffe(te, STATION)
skala = mase_skala(tr['y']); t0 = tr.sort_values('ds').iloc[-1]        # letzter TRAININGswert

m_wx  = fit_prophet(tr, REGS_WX)
m_all = fit_prophet(tr, REGS_ALL)
idx = te.set_index('ds').index

yhat = {
    'Prophet + Wetter (Lag, nur Train)':    predict_frozen(m_wx,  idx, t0, REGS_WX),
    'Prophet + Wetter+Co (Lag, nur Train)': predict_frozen(m_all, idx, t0, REGS_ALL),
}
o = te.sort_values('ds').reset_index(drop=True); yy = o['y'].to_numpy()
rows = []
for name, yh in [('Seasonal Naive', seasonal_naive(tr, te))] + list(yhat.items()):
    yh = pd.Series(np.asarray(yh), index=te['ds']).reindex(o['ds']).to_numpy()
    for h in HORIZONTE + [len(o)]:
        w = slice(0, h); hn = f'{h} h' if h < len(o) else 'ganzes Jahr'
        rows.append({'Modell': name, 'Horizont': hn, 'MAE': mae(yy[w], yh[w]),
                     'RMSE': rmse(yy[w], yh[w]), 'MASE': mae(yy[w], yh[w]) / skala,
                     'MAPE %': mape(yy[w], yh[w])})
tab_einzel = pd.DataFrame(rows)[['Modell', 'Horizont', 'MAE', 'RMSE', 'MASE', 'MAPE %']]
print(f'Einzelanalyse ({STATION}) - Wetter+Co als Lag (nur Trainingswerte):')
print(tab_einzel.round(3).to_string(index=False))
tab_einzel.to_csv('../data/ergebnis_coschadstoffe_einzel.csv', index=False)
print('\ngespeichert -> ../data/ergebnis_coschadstoffe_einzel.csv')

21:32:57 - cmdstanpy - INFO - Chain [1] start processing
21:33:08 - cmdstanpy - INFO - Chain [1] done processing
21:33:09 - cmdstanpy - INFO - Chain [1] start processing
21:33:29 - cmdstanpy - INFO - Chain [1] done processing


Einzelanalyse (Aotizhongxin) - Wetter+Co als Lag (nur Trainingswerte):
                              Modell    Horizont     MAE    RMSE  MASE  MAPE %
                      Seasonal Naive         8 h  86.750  88.659 1.448  88.203
                      Seasonal Naive        24 h  92.500  95.974 1.544  75.174
                      Seasonal Naive        48 h 114.896 125.722 1.917  75.804
                      Seasonal Naive        72 h 166.764 188.372 2.783  80.553
                      Seasonal Naive ganzes Jahr  63.590  98.566 1.061 120.641
   Prophet + Wetter (Lag, nur Train)         8 h  37.722  43.980 0.629  34.856
   Prophet + Wetter (Lag, nur Train)        24 h  61.937  70.552 1.034  45.636
   Prophet + Wetter (Lag, nur Train)        48 h  86.202  97.397 1.438  53.932
   Prophet + Wetter (Lag, nur Train)        72 h 137.834 163.573 2.300  62.200
   Prophet + Wetter (Lag, nur Train) ganzes Jahr  74.051 104.402 1.236 126.209
Prophet + Wetter+Co (Lag, nur Train)         8 h  30.828  35

## 12.5 Alle 12 Stationen — rollierend (Durchschnitts-Startpunkt)

Je Station ein Fit für **nur Wetter** und einer für **Wetter+Co**. An ~12 Startpunkten (alle 30 Tage) wird +1…+72 h vorhergesagt, wobei die Regressoren am jeweiligen Startpunkt eingefroren werden (letzter bekannter Wert, nie das Fenster). Gemittelt über alle Startpunkte und Stationen.

> ⏱️ **Laufzeit:** 12 Stationen × 2 Modelle = **24 Fits**, grob **10–15 Min**.

In [4]:
def profil_naive(hist, fds):
    prof = hist.tail(24).assign(h=lambda d: d['ds'].dt.hour).set_index('h')['y']
    return prof.reindex(fds.hour).to_numpy()

CUTOFF_STEP_D, MAX_H = 30, 72
rows = []
for st in stationen:
    tr, te = lade('behandelt', st)
    tr = mit_coschadstoffe(tr, st).sort_values('ds').reset_index(drop=True)
    te = mit_coschadstoffe(te, st).sort_values('ds').reset_index(drop=True)
    skala = mase_skala(tr['y'])
    m_wx = fit_prophet(tr, REGS_WX); m_all = fit_prophet(tr, REGS_ALL)
    full = pd.concat([tr[['ds', 'y'] + REGS_ALL], te[['ds', 'y'] + REGS_ALL]], ignore_index=True)
    tei = te.set_index('ds')
    cutoffs = pd.date_range(te['ds'].min(), te['ds'].max() - pd.Timedelta(hours=MAX_H), freq=f'{CUTOFF_STEP_D}D')
    lead=[]; yt=[]; ywx=[]; yall=[]; yn=[]
    for c0 in cutoffs:
        fds = pd.date_range(c0 + pd.Timedelta(hours=1), c0 + pd.Timedelta(hours=MAX_H), freq='h')
        fds = fds[fds.isin(tei.index)]
        if len(fds) == 0: continue
        sub = tei.loc[fds]; hist = full[full['ds'] <= c0]; w0 = hist.iloc[-1]   # letzter bekannter Wert
        fwx = pd.DataFrame({'ds': fds}); fal = pd.DataFrame({'ds': fds})
        for r in REGS_WX:  fwx[r] = w0[r]
        for r in REGS_ALL: fal[r] = w0[r]
        ywx.extend(m_wx.predict(fwx)['yhat'].clip(lower=0).to_numpy())
        yall.extend(m_all.predict(fal)['yhat'].clip(lower=0).to_numpy())
        yn.extend(profil_naive(hist, fds)); yt.extend(sub['y'].to_numpy())
        lead.extend(((fds - c0) / pd.Timedelta(hours=1)).astype(int))
    d = pd.DataFrame({'lead': lead, 'y': yt, 'wx': ywx, 'all': yall, 'naive': yn}).dropna()
    for b in HORIZONTE:
        w = d[d['lead'] <= b]
        for name, col in [('Seasonal Naive', 'naive'), ('Prophet + Wetter (Lag)', 'wx'),
                          ('Prophet + Wetter+Co (Lag)', 'all')]:
            rows.append({'Station': st, 'Modell': name, 'Horizont': f'{b} h',
                         'MAE': mae(w['y'], w[col]), 'RMSE': rmse(w['y'], w[col]),
                         'MASE': mae(w['y'], w[col]) / skala, 'MAPE %': mape(w['y'], w[col])})
    print('fertig:', st)

roll = pd.DataFrame(rows)
agg = roll.groupby(['Modell', 'Horizont'])[['MAE', 'RMSE', 'MASE', 'MAPE %']].mean().reset_index()
agg['ord'] = agg['Horizont'].map({'8 h': 0, '24 h': 1, '48 h': 2, '72 h': 3})
agg = agg.sort_values(['Modell', 'ord']).drop(columns='ord').round(3)
print('\nRollierend, Mittel ueber 12 Stationen & Startpunkte (Lag):')
print(agg.to_string(index=False))
agg.to_csv('../data/ergebnis_coschadstoffe_rolling_allstations.csv', index=False)
print('\ngespeichert -> ../data/ergebnis_coschadstoffe_rolling_allstations.csv')

21:33:34 - cmdstanpy - INFO - Chain [1] start processing
21:33:44 - cmdstanpy - INFO - Chain [1] done processing
21:33:46 - cmdstanpy - INFO - Chain [1] start processing
21:34:07 - cmdstanpy - INFO - Chain [1] done processing


fertig: Aotizhongxin


21:34:11 - cmdstanpy - INFO - Chain [1] start processing
21:34:24 - cmdstanpy - INFO - Chain [1] done processing
21:34:26 - cmdstanpy - INFO - Chain [1] start processing
21:34:45 - cmdstanpy - INFO - Chain [1] done processing


fertig: Changping


21:34:49 - cmdstanpy - INFO - Chain [1] start processing
21:35:02 - cmdstanpy - INFO - Chain [1] done processing
21:35:04 - cmdstanpy - INFO - Chain [1] start processing
21:35:25 - cmdstanpy - INFO - Chain [1] done processing


...

fertig: Tiantan


21:40:44 - cmdstanpy - INFO - Chain [1] start processing
21:40:58 - cmdstanpy - INFO - Chain [1] done processing
21:41:00 - cmdstanpy - INFO - Chain [1] start processing
21:41:17 - cmdstanpy - INFO - Chain [1] done processing


fertig: Wanliu


21:41:21 - cmdstanpy - INFO - Chain [1] start processing
21:41:39 - cmdstanpy - INFO - Chain [1] done processing
21:41:41 - cmdstanpy - INFO - Chain [1] start processing
21:42:11 - cmdstanpy - INFO - Chain [1] done processing


fertig: Wanshouxigong

Rollierend, Mittel ueber 12 Stationen & Startpunkte (Lag):
                   Modell Horizont    MAE   RMSE  MASE  MAPE %
   Prophet + Wetter (Lag)      8 h 60.097 79.289 1.022  83.818
   Prophet + Wetter (Lag)     24 h 70.330 88.931 1.199 157.048
   Prophet + Wetter (Lag)     48 h 68.335 87.901 1.162 221.983
   Prophet + Wetter (Lag)     72 h 64.048 86.474 1.088 229.985
Prophet + Wetter+Co (Lag)      8 h 33.468 45.036 0.569  78.370
Prophet + Wetter+Co (Lag)     24 h 49.608 64.852 0.846 169.551
Prophet + Wetter+Co (Lag)     48 h 60.098 80.649 1.023 307.150
Prophet + Wetter+Co (Lag)     72 h 65.523 89.233 1.114 333.135
           Seasonal Naive      8 h 66.842 83.676 1.132 102.617
           Seasonal Naive     24 h 66.947 85.151 1.140 160.402
           Seasonal Naive     48 h 68.573 90.512 1.168 232.371
           Seasonal Naive     72 h 66.214 92.441 1.127 238.192

gespeichert -> ../data/ergebnis_coschadstoffe_rolling_allstations.csv
